# MoE 详细学习笔记：从 1991 原始 Mixture of Experts 开始

这份笔记主要基于 Jacobs、Jordan、Nowlan、Hinton 在 1991 年发表的 **《Adaptive Mixtures of Local Experts》**。论文的核心不是今天 Transformer 里“省算力”的 Sparse MoE，而是：

$$
\boxed{\text{如何让多个网络自动竞争、分工，并分别学习不同子任务}}
$$

论文认为，当一个网络同时学习多个不同子任务时，梯度之间可能产生 interference；因此提出多个 local expert 加一个 gating network，让系统自己学习“什么样的输入应该由哪个 expert 负责”。论文用一个 **speaker-independent vowel discrimination** 任务展示了这种自动任务分解。:chatgpt-content-reference{index="0"}

[原论文 PDF](https://www.cs.toronto.edu/~hinton/absps/jjnh91.pdf?utm_source=chatgpt.com)

---

## 1. 为什么要提出 MoE？

假设训练集中实际上存在多个不同的子任务。

传统方法是用一个网络：

$$
x\rightarrow f(x)\rightarrow y
$$

同时学习所有样本。

问题在于，不同子任务可能希望网络参数向不同方向更新，于是产生：

$$
\boxed{\text{interference}}
$$

论文希望改成：

$$
x
\rightarrow
\begin{cases}
E_1(x)\\
E_2(x)\\
\vdots\\
E_K(x)
\end{cases}
$$

不同 Expert 分别负责训练集的不同局部区域。

但是这里立刻产生一个问题：

> **我们事先不知道哪个样本属于哪个子任务。**

因此还需要一个：

$$
\boxed{\text{Gating Network}}
$$

它负责根据 \(x\) 判断：

$$
\text{当前样本应该交给哪个 Expert。}
$$

最终形成：

$$
\boxed{
\text{Mixture of Experts}
=
\text{Gating Network}
+
\text{多个 Local Experts}
}
$$

原论文特别强调，希望这种任务分解是**学习出来的，而不是人工提前给 Expert 标注任务类型**。:chatgpt-content-reference{index="2"}

---

# 2. 原始 MoE 的网络结构

论文 Figure 1 的结构非常直接：所有 Expert 和 gating network 接收相同输入；每个 Expert 输出自己的结果，gate 输出归一化的 mixing proportions。论文还从生成模型角度把 gate 看成一个 stochastic selector。:chatgpt-content-reference{index="3"}

对于输入：

$$
x
$$

第 \(i\) 个 Expert：

$$
\boxed{o_i=f_i(x;\theta_i)}
$$

Gating Network：

$$
a=g(x;\theta_G)
$$

得到 logits：

$$
a=[a_1,a_2,\cdots,a_K]
$$

经过 Softmax：

$$
\boxed{
p_i=
\frac{\exp(a_i)}
{\sum_j\exp(a_j)}
}
$$

满足：

$$
p_i\geq0,\qquad
\sum_i p_i=1
$$

因此可以把：

$$
\boxed{p_i=P(z=i|x)}
$$

理解成：

> 给定输入 \(x\)，选择 Expert \(i\) 的概率。

其中：

$$
z\in\{1,\dots,K\}
$$

是一个隐藏变量，表示“哪个 Expert 负责当前样本”。

---

# 3. 为什么不能简单把所有 Expert 输出加权平均？

这是理解原论文最重要的动机之一。

一种非常自然的想法是：

$$
\hat y=\sum_i p_i o_i
$$

然后普通平方误差：

$$
\boxed{
E=
\left\|
d-\sum_i p_i o_i
\right\|^2
}
$$

其中 \(d\) 是 desired output。

但论文指出，这种 loss 会让 Expert 倾向于**合作（cooperate）**，而不是**竞争（compete）**。因为系统只要求多个 Expert 的加权平均正确，而不要求某一个 Expert 自己把样本预测好。:chatgpt-content-reference{index="4"}

例如：

$$
d=10
$$

两个 Expert：

$$
o_1=5,\qquad o_2=15
$$

Gate：

$$
p_1=p_2=0.5
$$

最终：

$$
\hat y
=
0.5\times5+0.5\times15
=
10
$$

所以：

$$
E=0
$$

但实际上：

$$
o_1\neq10,\qquad o_2\neq10
$$

没有任何一个 Expert 真正会解决这个样本。

这不是作者想要的结果。

作者真正希望的是：

> 某个 Expert 一旦比其他 Expert 更擅长某类样本，就应该让它获得更多这类样本，最终形成 specialization。

---

# 4. 第一步改进：不要平均 Expert 的输出

论文于是考虑：

> Gate 不是把 Expert 输出平均，而是随机选择一个 Expert。

即：

$$
z\sim Categorical(p_1,\dots,p_K)
$$

如果：

$$
z=i
$$

那么直接使用：

$$
o_i
$$

因此平方误差的期望变成：

$$
\boxed{
E
=
\sum_i
p_i
\|d-o_i\|^2
}
$$

这样，每个 Expert 必须自己接近 target，而不能依靠其他 Expert 抵消自己的错误。论文 Eq. 1.2 正是这个思路。:chatgpt-content-reference{index="5"}

但论文随后又提出了一个更好的 probabilistic loss。

---

# 5. 从 Competitive Learning 引入概率解释

这是原论文很重要的一步。

作者首先回顾 soft competitive learning：

每个竞争单元代表一个多维 Gaussian 的均值。

生成一个输出的过程可以理解成：

第一步：

$$
z\sim Categorical(p_1,\dots,p_K)
$$

第二步：

$$
o\sim
\mathcal N(\mu_z,\Sigma_z)
$$

因此整个输出分布是：

$$
p(o)
=
\sum_i
p_i
\mathcal N(o;\mu_i,\Sigma_i)
$$

这就是 mixture distribution。

随后作者把普通 competitive-learning unit 替换成整个 Expert Network：

$$
\boxed{
\mu_i
\longrightarrow
o_i(x)
}
$$

也就是说，现在 Gaussian 的均值不再是一个固定参数，而是：

$$
\boxed{
o_i=f_i(x)
}
$$

同时 mixing proportion 也不再固定，而是由 gating network 根据输入产生：

$$
\boxed{
p_i=p_i(x)
}
$$

论文明确描述了这种解释：Expert 输出向量指定一个多维 Gaussian 的均值，而 gate 根据输入决定各 Expert 的 mixing proportions。:chatgpt-content-reference{index="6"}

---

# 6. Expert 到底“输出 Gaussian”吗？

不是。

Expert 神经网络实际只输出：

$$
\boxed{o_i}
$$

我们把它**解释成 Gaussian 的均值**：

$$
\mu_i=o_i
$$

然后定义：

$$
p(d|x,z=i)
=
\mathcal N(d;o_i,\Sigma)
$$

因此：

$$
\boxed{
Expert网络输出均值
\rightarrow
利用Gaussian解析式计算target位置的density
}
$$

Expert 并不需要真的输出一整条 Gaussian 曲线。

如果简单假设：

$$
\Sigma=I
$$

那么：

$$
p(d|x,z=i)
\propto
\exp\left(
-\frac12\|d-o_i\|^2
\right)
$$

为了后面简写，定义：

$$
\boxed{
q_i
\equiv
p(d|x,z=i)
}
$$

那么：

$$
q_i
\propto
e^{-\frac12\|d-o_i\|^2}
$$

---

# 7. Gaussian 在这里有什么意义？

它实际上是把：

$$
\boxed{\text{Expert的预测误差}}
$$

变成：

$$
\boxed{\text{Expert产生target的likelihood}}
$$

因为：

$$
q_i
\propto
e^{-\frac12\|d-o_i\|^2}
$$

所以：

$$
\|d-o_i\|^2\downarrow
\Rightarrow
q_i\uparrow
$$

也就是说：

> Expert 越接近 target，这个 Expert 对当前训练样本的 likelihood 越大。

另外要严格区分：

$$
q_i
$$

对于连续输出是**概率密度**，不是“恰好输出 \(d\)”的单点概率。

---

# 8. 整个 MoE 的输出分布

现在已经有两个部分。

Gate：

$$
\boxed{
P(z=i|x)=p_i
}
$$

Expert：

$$
\boxed{
p(d|x,z=i)=q_i
}
$$

但是 \(z\) 是隐藏变量。

我们不知道真实样本到底是由哪个 Expert 负责。

因此根据全概率公式，把 \(z\) marginalize：

$$
p(d|x)
=
\sum_i
p(d,z=i|x)
$$

又因为：

$$
p(d,z=i|x)
=
P(z=i|x)
p(d|x,z=i)
$$

所以：

$$
\boxed{
p(d|x)
=
\sum_i p_iq_i
}
$$

Gaussian 情况就是：

$$
\boxed{
p(d|x)
=
\sum_i
p_i(x)
\mathcal N(d;o_i(x),\Sigma)
}
$$

这就是整个 MoE 的 conditional mixture distribution。

---

# 9. 总训练目标

既然模型定义了：

$$
p(d|x)
$$

最自然的 Maximum Likelihood 训练就是：

$$
\max p(d|x)
$$

或者：

$$
\max\log p(d|x)
$$

因此最小化 Negative Log-Likelihood：

$$
\boxed{
L_{\text{total}}
=
-\log
\sum_i p_iq_i
}
$$

如果使用论文 Eq. 1.3 对应的单位方差形式：

$$
\boxed{
L_{\text{total}}
=
-\log
\sum_i
p_i
\exp
\left(
-\frac12\|d-o_i\|^2
\right)
}
$$

这就是核心。

论文指出 Eq. 1.3 可以解释为 Gaussian mixture 下生成 desired output vector 的 negative log probability，并发现它比前面的期望平方误差形式表现更好。:chatgpt-content-reference{index="7"}

---

# 10. Responsibility 从哪里来？

现在引入最关键的量。

Gate 在没看到 target 前输出：

$$
p_i=P(z=i|x)
$$

但是训练阶段我们已经知道：

$$
x,d
$$

所以可以问：

> 在已经看到真实 \(d\) 的情况下，这个样本究竟应该由 Expert \(i\) 负责多少？

也就是：

$$
P(z=i|x,d)
$$

根据 Bayes：

$$
P(z=i|x,d)
=
\frac{
p(d|x,z=i)P(z=i|x)
}{
p(d|x)
}
$$

代入：

$$
P(z=i|x)=p_i
$$

$$
p(d|x,z=i)=q_i
$$

以及：

$$
p(d|x)=\sum_jp_jq_j
$$

得到：

$$
\boxed{
r_i
\equiv
P(z=i|x,d)
=
\frac{
p_iq_i
}{
\sum_jp_jq_j
}
}
$$

这就是我们一直说的：

$$
\boxed{\text{Responsibility}}
$$

---

# 11. \(p_i\) 和 \(r_i\) 的区别

这个必须非常清楚。

### Gate 输出

$$
\boxed{
p_i=P(z=i|x)
}
$$

只看 \(x\)。

可以理解成：

> **在还没有看到答案之前，Gate 认为 Expert \(i\) 应该负责多少。**

### Responsibility

$$
\boxed{
r_i=P(z=i|x,d)
}
$$

已经知道真实 target \(d\)，并且知道所有 Expert 实际预测得怎么样。

所以：

> **结合真实答案以后，重新判断 Expert \(i\) 应该负责多少。**

因此：

$$
\boxed{
p_i
\xrightarrow[\text{Expert实际表现}]{d}
r_i
}
$$

本质上就是一次 Bayesian update。

---

# 12. Responsibility 为什么要“归一化”？

定义：

$$
s_i=p_iq_i
$$

这是未归一化的 joint score：

$$
s_i=p(d,z=i|x)
$$

所有 Expert 加起来：

$$
\sum_j s_j
=
\sum_jp_jq_j
=
p(d|x)
$$

因此：

$$
r_i
=
\frac{s_i}{\sum_js_j}
$$

自然满足：

$$
\sum_i r_i=1
$$

所以归一化不是人为加进去的技巧，而是 Bayes 公式里的：

$$
\boxed{
\frac{\text{joint}}
{\text{marginal}}
}
$$

---

# 13. Gaussian 情况下的 Responsibility

如果：

$$
q_i
\propto
\exp\left(
-\frac12\|d-o_i\|^2
\right)
$$

那么：

$$
\boxed{
r_i
=
\frac{
p_i
e^{-\frac12\|d-o_i\|^2}
}{
\sum_j
p_j
e^{-\frac12\|d-o_j\|^2}
}
}
$$

进一步写成：

$$
\boxed{
r_i
=
Softmax_i
\left(
\log p_i
-\frac12\|d-o_i\|^2
\right)
}
$$

这个公式非常值得记。

它告诉我们：

$$
\boxed{
\text{Responsibility}
=
\text{Gate的先验偏好}
+
\text{Expert的实际预测质量}
}
$$

Expert 预测得越准：

$$
\|d-o_i\|^2\downarrow
$$

则：

$$
r_i\uparrow
$$

---

# 14. 最核心的问题：一个总 Loss 怎么同时训练 Gate 和 Expert？

现在我们来到整个算法最重要的数学部分。

总 Loss：

$$
\boxed{
L
=
-\log
\sum_i p_iq_i
}
$$

它的输入实际上来自两个分支：

$$
\underbrace{a_i}_{Gate\ logits}
\rightarrow
p_i=Softmax(a)_i
$$

以及：

$$
\underbrace{o_i}_{Expert\ output}
\rightarrow
q_i
$$

所以可以写成：

$$
\boxed{
L=L(a,o)
}
$$

因此反向传播自然产生两组梯度：

$$
\boxed{
\frac{\partial L}{\partial a_i}
}
$$

训练 Gate；

以及：

$$
\boxed{
\frac{\partial L}{\partial o_i}
}
$$

训练 Expert。

**我们不是先人为设计两个 loss。**

而是：

$$
\boxed{
L_{\text{total}}
\rightarrow
\begin{cases}
\nabla_aL & Gate\\
\nabla_oL & Expert
\end{cases}
}
$$

然后发现这两个梯度分别可以解释成两个非常熟悉的等价 loss。

---

# 15. 先推 Expert 的梯度

定义：

$$
S=\sum_jp_jq_j
$$

所以：

$$
L=-\log S
$$

对第 \(i\) 个 Expert 输出 \(o_i\) 求导：

$$
\frac{\partial L}{\partial o_i}
=
-\frac1S
\frac{\partial S}{\partial o_i}
$$

因为只有：

$$
q_i
$$

依赖 \(o_i\)，所以：

$$
\frac{\partial S}{\partial o_i}
=
p_i
\frac{\partial q_i}{\partial o_i}
$$

因此：

$$
\frac{\partial L}{\partial o_i}
=
-\frac{p_i}{S}
\frac{\partial q_i}{\partial o_i}
$$

利用：

$$
\frac{\partial q_i}{\partial o_i}
=
q_i
\frac{\partial\log q_i}{\partial o_i}
$$

得到：

$$
\frac{\partial L}{\partial o_i}
=
-\frac{p_iq_i}{S}
\frac{\partial\log q_i}{\partial o_i}
$$

而：

$$
\frac{p_iq_i}{S}=r_i
$$

所以：

$$
\boxed{
\frac{\partial L}{\partial o_i}
=
-r_i
\frac{\partial\log q_i}{\partial o_i}
}
$$

这就是第一个核心结论。

---

# 16. 代入 Gaussian

单位方差 Gaussian：

$$
q_i
\propto
e^{-\frac12\|d-o_i\|^2}
$$

因此：

$$
-\log q_i
=
\frac12\|d-o_i\|^2+C
$$

求导：

$$
\frac{\partial(-\log q_i)}
{\partial o_i}
=
o_i-d
$$

因此：

$$
\boxed{
\frac{\partial L}
{\partial o_i}
=
r_i(o_i-d)
}
$$

如果保留一般方差：

$$
\boxed{
\frac{\partial L}
{\partial o_i}
=
r_i\frac{o_i-d}{\sigma^2}
}
$$

这就是为什么 responsibility 会自动控制 Expert 的训练强度。

---

# 17. 从梯度反推等价 Expert Loss

我们发现总 Loss 给 Expert 的梯度是：

$$
r_i(o_i-d)
$$

现在问：

> 什么 loss 会产生相同梯度？

考虑：

$$
\boxed{
L_E
=
\sum_i
r_i
\frac12
\|d-o_i\|^2
}
$$

如果把 \(r_i\) 暂时视为常量：

$$
\frac{\partial L_E}
{\partial o_i}
=
r_i(o_i-d)
$$

恰好与总 Loss 相同：

$$
\boxed{
\nabla_oL_{\text{total}}
=
\nabla_oL_E
}
$$

所以从 Expert 的角度：

$$
\boxed{
L_E
=
\sum_i
r_i
\frac12\|d-o_i\|^2
}
$$

就是一个**等价的 responsibility-weighted regression loss**。

更一般地：

$$
\boxed{
L_E
=
-\sum_i r_i\log q_i
}
$$

---

# 18. 再推 Gate 的梯度

Gate 网络首先产生 logits：

$$
a_i
$$

然后：

$$
p_i
=
\frac{e^{a_i}}
{\sum_ke^{a_k}}
$$

我们要求：

$$
\frac{\partial L}{\partial a_i}
$$

还是：

$$
L=-\log S
$$

其中：

$$
S=\sum_jp_jq_j
$$

首先：

$$
\frac{\partial L}{\partial p_j}
=
-\frac{q_j}{S}
$$

Softmax 导数：

$$
\frac{\partial p_j}{\partial a_i}
=
p_j(\delta_{ij}-p_i)
$$

根据链式法则：

$$
\frac{\partial L}{\partial a_i}
=
\sum_j
\frac{\partial L}{\partial p_j}
\frac{\partial p_j}{\partial a_i}
$$

代入：

$$
=
-\frac1S
\sum_j
q_jp_j
(\delta_{ij}-p_i)
$$

展开：

$$
=
-\frac1S
\sum_jq_jp_j\delta_{ij}
+
\frac{p_i}{S}
\sum_jq_jp_j
$$

第一项：

$$
-\frac{q_ip_i}{S}
$$

第二项因为：

$$
\sum_jq_jp_j=S
$$

所以：

$$
\frac{p_i}{S}S=p_i
$$

因此：

$$
\boxed{
\frac{\partial L}
{\partial a_i}
=
p_i-\frac{p_iq_i}{S}
}
$$

而：

$$
r_i=\frac{p_iq_i}{S}
$$

所以最终：

$$
\boxed{
\frac{\partial L}
{\partial a_i}
=
p_i-r_i
}
$$

这是整个 MoE 最值得记住的公式之一。

---

# 19. 从 Gate 梯度反推等价 Gate Loss

现在考虑一个 soft-label Cross Entropy：

$$
\boxed{
L_G
=
-\sum_i
r_i\log p_i
}
$$

如果：

$$
p=Softmax(a)
$$

并且把 \(r\) 看作固定 target，那么标准 Softmax + CE 梯度：

$$
\boxed{
\frac{\partial L_G}{\partial a_i}
=
p_i-r_i
}
$$

正好：

$$
\boxed{
\nabla_aL_{\text{total}}
=
\nabla_aL_G
}
$$

所以从 Gate 的优化角度：

$$
\boxed{
L_G
=
-\sum_i
r_i\log p_i
}
$$

可以理解成：

> 用 posterior responsibility \(r\) 作为 soft target，训练 Gate 的输出 \(p\)。

---

# 20. 所以一个总 Loss 自动产生了两个学习任务

现在整个数学关系非常漂亮：

$$
\boxed{
L_{\text{total}}
=
-\log\sum_i p_iq_i
}
$$

对 Gate：

$$
\boxed{
\frac{\partial L}{\partial a_i}
=
p_i-r_i
}
$$

等价于：

$$
\boxed{
L_G=-\sum_i r_i\log p_i
}
$$

对 Expert：

$$
\boxed{
\frac{\partial L}{\partial o_i}
=
r_i(o_i-d)
}
$$

等价于：

$$
\boxed{
L_E=
\sum_i
r_i\frac12\|d-o_i\|^2
}
$$

因此：

$$
\boxed{
\begin{array}{c}
L_{\text{total}}\\
\downarrow\\
r_i=P(z=i|x,d)\\
\swarrow\qquad\searrow\\
Gate\qquad Expert\\
p_i\rightarrow r_i
\qquad
r_i\times\text{prediction error}
\end{array}
}
$$

这就是 Router/Gate 和 Expert 协同训练的数学来源。

---

# 21. 那么 \(L_{\text{total}}\) 是不是 \(L_G+L_E\)？

这里一定要严谨。

不能简单写：

$$
L_{\text{total}}=L_G+L_E
$$

还差一项。

因为：

$$
r_i=\frac{p_iq_i}{S}
$$

所以：

$$
S=\frac{p_iq_i}{r_i}
$$

取 log：

$$
\log S
=
\log p_i+\log q_i-\log r_i
$$

乘 \(r_i\) 并求和：

$$
\log S
=
\sum_i
r_i
[
\log p_i+\log q_i-\log r_i
]
$$

因此：

$$
-\log S
=
-\sum_i r_i\log p_i
-\sum_i r_i\log q_i
+\sum_i r_i\log r_i
$$

于是：

$$
\boxed{
L_{\text{total}}
=
L_G
+
L_E
-
H(r)
}
$$

其中：

$$
H(r)
=
-\sum_i r_i\log r_i
$$

是 responsibility 的 entropy。

---

# 22. 为什么又可以用 \(L_G+L_E\) 理解？

关键在于：

$$
\boxed{\text{固定 }r}
$$

也就是计算出当前 posterior：

$$
r_i
=
\frac{p_iq_i}
{\sum_jp_jq_j}
$$

然后把它当作 target，不再让梯度穿过 \(r\)。

这时候：

$$
H(r)
$$

相对于当前参数更新就是常数。

因此：

$$
\arg\min
L_{\text{total}}
$$

与优化：

$$
\boxed{
L_G+L_E
}
$$

在这种 fixed-\(r\) 的解释下具有相应的等价梯度结构。

这也是为什么它和 mixture model / EM 的思想非常接近：

$$
\boxed{
\text{先计算谁负责}
\rightarrow
\text{再根据责任度更新模型}
}
$$

但原论文这里直接采用 gradient descent 训练，并不需要把实现理解成必须显式执行标准 GMM 的闭式 EM。论文实验说明使用简单的固定步长 gradient descent，没有 momentum 或其他 acceleration。:chatgpt-content-reference{index="8"}

---

# 23. 为什么这样会让 Expert 自动专业化？

现在假设某个训练样本上：

$$
p_1=p_2
$$

也就是说一开始 Gate 没有明显偏好。

但是 Expert 1：

$$
\|d-o_1\|^2
<
\|d-o_2\|^2
$$

于是：

$$
q_1>q_2
$$

所以：

$$
r_1>r_2
$$

接下来同时发生两件事情。

Expert 侧：

$$
\frac{\partial L}{\partial o_1}
=
r_1(o_1-d)
$$

因为：

$$
r_1\text{ 更大}
$$

所以 Expert 1 在这个样本上获得更强的学习信号。

Gate 侧：

$$
\frac{\partial L}{\partial a_1}
=
p_1-r_1
$$

由于：

$$
r_1>p_1
$$

所以：

$$
p_1\uparrow
$$

也就是说 Gate 以后看到类似输入，更倾向选择 Expert 1。

因此：

$$
\boxed{
E_1稍微更擅长该类样本
}
$$

导致：

$$
q_1\uparrow
$$

导致：

$$
r_1\uparrow
$$

然后：

$$
\begin{cases}
E_1获得更多该类训练\\
Gate更倾向把该类输入交给E_1
\end{cases}
$$

于是 Expert 1：

$$
\boxed{\text{越来越擅长这类输入}}
$$

这就是正反馈式的：

$$
\boxed{\text{specialization}}
$$

论文明确指出，新误差函数会提高表现优于加权平均的 Expert 的 responsibility，并降低表现较差 Expert 的 responsibility，从而倾向于让单个 Expert 对一个训练 case 形成主导。:chatgpt-content-reference{index="9"}

---

# 24. 原论文的 vowel discrimination 实验

现在回到原论文真正的例子。

任务是：

$$
\boxed{\text{speaker-independent four-class vowel discrimination}}
$$

四个 vowel：

$$
[i],\quad[I],\quad[a],\quad[\Lambda]
$$

数据来自：

$$
75
$$

位男性、女性和儿童 speaker。

输入只有两个特征：

$$
\boxed{
x=(F_1,F_2)
}
$$

分别是：

- first formant；
- second formant。

这些 formant 值被缩放后形成二维输入空间。:chatgpt-content-reference{index="10"}

---

# 25. 这个任务为什么适合 MoE？

数据在二维空间里形成两个主要的 overlapping pair：

$$
[i]\quad\text{和}\quad[I]
$$

以及：

$$
[a]\quad\text{和}\quad[\Lambda]
$$

所以整体分类边界不是简单一条直线就能很好解决。

但如果把问题拆开：

> 某些 Expert 专门处理一部分输入空间/类别边界，

那么每个 Expert 可以非常简单。

论文中的 Expert 被限制得很弱，每个 Expert 的结构使其只能形成**线性 decision surface**；但是多个 Expert + Gate 组合以后，可以完成整体更复杂的分类边界。:chatgpt-content-reference{index="11"}

这就是：

$$
\boxed{
\text{复杂全局问题}
\rightarrow
\text{多个简单局部问题}
}
$$

---

# 26. Expert 的“专业”不是人工规定的

这点尤其重要。

训练之前没有规定：

$$
Expert_1=[i]\text{ vs }[I]
$$

也没有规定：

$$
Expert_2=[a]\text{ vs }[\Lambda]
$$

而是通过前面那套竞争机制自己形成：

$$
\text{哪个Expert在某一区域稍微表现更好}
$$

↓

$$
r_i\uparrow
$$

↓

$$
\text{得到更多该区域的学习}
$$

↓

$$
\text{Gate也更倾向选择它}
$$

↓

$$
\boxed{\text{自动形成task decomposition}}
$$

论文的 Figure 2 就展示了 Expert decision lines 和 gating decision line 如何共同把输入空间分解；作者指出不同 Expert 学会集中处理两组 overlapping classes。:chatgpt-content-reference{index="12"}

---

# 27. 原论文最终分类时怎么做？

这里也要区分我们前面用于解释训练的 Gaussian mixture 和最终 vowel classification。

论文的分类判定中，每个 Expert 给出关于类别的输出/概率分布，然后按照 Gate 给出的 mixing proportions 组合：

$$
\boxed{
P(c|x)
=
\sum_i
p_i(x)P_i(c|x)
}
$$

最后：

$$
\boxed{
\hat c
=
\arg\max_cP(c|x)
}
$$

也就是说，最终分类使用的是多个 Expert 的类别分布按照 gate proportion 混合，再选择最可能类别。论文对此有明确说明。:chatgpt-content-reference{index="13"}

因此不要把：

$$
\text{Gaussian概率解释}
$$

误解成：

> vowel 实验最后一定从某个 Gaussian 随机采一个数作为类别。

不是这个意思。

Gaussian mixture 是理解 Eq. 1.3 概率训练目标的重要生成模型解释。

---

# 28. 原论文实验结果说明什么？

论文比较了：

- 4 Experts；
- 8 Experts；
- 单隐藏层 BP 网络，6 hidden units；
- 单隐藏层 BP 网络，12 hidden units。

论文报告中这些系统最终训练正确率约为 88%、测试正确率约为 90%，但 MoE 达到规定误差标准所需的平均 epoch 明显更少：4 Experts 约 1124，8 Experts 约 1083，而两个 BP 基线约 2209 和 2435。作者据此强调 mixture-of-experts 在这个任务上的学习速度优势，以及随着 Expert 数量增加仍能很好扩展。:chatgpt-content-reference{index="14"}

这里核心不是：

$$
\text{Expert越多准确率一定越高}
$$

而是展示：

$$
\boxed{
\text{通过自动task decomposition减少不同子任务之间的interference}
}
$$

可以明显改善学习过程。

---

# 29. 整个原始 MoE 算法串起来

对于每一个训练样本：

$$
(x,d)
$$

**第一步，所有 Expert 前向：**

$$
o_i=E_i(x)
$$

**第二步，Gate 前向：**

$$
a=G(x)
$$

$$
p_i=Softmax(a)_i
$$

**第三步，计算每个 Expert 对 target 的 likelihood：**

$$
q_i
=
\mathcal N(d;o_i,\Sigma)
$$

在论文 Eq. 1.3 的简化形式：

$$
q_i
\propto
e^{-\frac12\|d-o_i\|^2}
$$

**第四步，计算 mixture likelihood：**

$$
\boxed{
p(d|x)=\sum_i p_iq_i
}
$$

**第五步，总 Loss：**

$$
\boxed{
L
=
-\log
\sum_i p_iq_i
}
$$

**第六步，可以显式理解 responsibility：**

$$
\boxed{
r_i
=
\frac{p_iq_i}
{\sum_jp_jq_j}
}
$$

**第七步，直接反向传播总 Loss。**

Gate 得到：

$$
\boxed{
\frac{\partial L}{\partial a_i}
=
p_i-r_i
}
$$

Expert 得到：

$$
\boxed{
\frac{\partial L}{\partial o_i}
=
r_i(o_i-d)
}
$$

然后继续通过链式法则更新各自参数：

$$
\frac{\partial L}{\partial\theta_G}
=
\sum_i
\frac{\partial L}{\partial a_i}
\frac{\partial a_i}{\partial\theta_G}
$$

以及：

$$
\frac{\partial L}{\partial\theta_i}
=
\frac{\partial L}{\partial o_i}
\frac{\partial o_i}{\partial\theta_i}
$$

---

# 30. 最后把整个 MoE 数学逻辑压缩成 5 个公式

如果面试或者复习，只记下面五个基本就够了。

### ① Gate

$$
\boxed{
p_i=P(z=i|x)=Softmax(a)_i
}
$$

### ② Expert likelihood

$$
\boxed{
q_i=p(d|x,z=i)
\propto
e^{-\frac12\|d-o_i\|^2}
}
$$

### ③ 总 Loss

$$
\boxed{
L
=
-\log
\sum_i p_iq_i
}
$$

### ④ Responsibility

$$
\boxed{
r_i
=
P(z=i|x,d)
=
\frac{p_iq_i}
{\sum_jp_jq_j}
}
$$

### ⑤ 两条最核心梯度

Gate：

$$
\boxed{
\frac{\partial L}{\partial a_i}
=
p_i-r_i
}
$$

Expert：

$$
\boxed{
\frac{\partial L}{\partial o_i}
=
r_i(o_i-d)
}
$$

于是：

$$
\boxed{
\text{Gate学习“谁应该负责”}
}
$$

同时：

$$
\boxed{
\text{Expert按照自己的responsibility学习target}
}
$$

同一个 \(r_i\) 把二者耦合起来，于是：

$$
\boxed{
\text{竞争}
\rightarrow
\text{分工}
\rightarrow
\text{专业化}
}
$$

这就是 **1991 原始 Mixture of Experts 最核心的思想**。

---

## 31. 和现代 MoE 先不要混在一起

学到这里，可以先把原始 MoE 定义成：

$$
\boxed{
\text{Input-dependent mixture model}
}
$$

它最初解决的核心问题是：

$$
\boxed{\text{task decomposition + reducing interference}}
$$

而今天 Transformer 里的 Sparse MoE 在这个思想基础上进一步发展出了：

$$
\text{token-level routing}
$$

$$
\text{Top-K experts}
$$

$$
\text{只计算被选中的 experts}
$$

$$
\text{load balancing}
$$

从而把 MoE 又变成了一种：

$$
\boxed{\text{扩大参数量但控制计算量}}
$$

的方法。

但这是**后面的演化**。理解现代 MoE 前，最重要的就是把这篇 1991 论文中的这条链真正吃透：

$$
\boxed{
p_i
\rightarrow
q_i
\rightarrow
\sum_i p_iq_i
\rightarrow
L
\rightarrow
r_i
\rightarrow
\begin{cases}
p_i-r_i\\
r_i(o_i-d)
\end{cases}
\rightarrow
\text{specialization}
}
$$

这条链就是整个原始 MoE 的数学骨架。

我们专门做一个**能直观看出 Expert 自动分工**的实验。

任务设为：

$$
y=
\begin{cases}
2x+1,&x<0\\
-3x+1,&x\geq0
\end{cases}
$$

设置 **2 个线性 Expert + 1 个 Gate**。训练过程中**不给 Expert 任何分工标签**，只优化 mixture NLL。训练完成后，在推理阶段分别打印两个 Expert 的输出、Router 概率和最终选择的 Expert，从而验证是否自动形成专业化。


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)


# ============================================================
# 1. Expert
# 每个 Expert 都是一个简单的线性模型：
# mu_i = w_i * x + b_i
# ============================================================
class Expert(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(1, 1)

    def forward(self, x):
        return self.fc(x)


# ============================================================
# 2. Gate / Router
# 根据输入 x 输出两个 Expert 的选择概率
#
# logits -> softmax -> [p1, p2]
# ============================================================
class Gate(nn.Module):
    def __init__(self, num_experts=2):
        super().__init__()
        self.fc = nn.Linear(1, num_experts)

    def forward(self, x):
        logits = self.fc(x)
        probs = F.softmax(logits, dim=-1)
        return logits, probs


# ============================================================
# 3. Mixture of Experts
# ============================================================
class MoE(nn.Module):
    def __init__(self, num_experts=2):
        super().__init__()

        self.num_experts = num_experts

        self.gate = Gate(num_experts)

        self.experts = nn.ModuleList([
            Expert() for _ in range(num_experts)
        ])

    def forward(self, x):

        # ---------- Router ----------
        logits, probs = self.gate(x)       # [B, K]

        # ---------- Experts ----------
        outputs = []

        for expert in self.experts:
            outputs.append(expert(x))      # 每个都是 [B,1]

        # [B,1] * K -> [B,K]
        mus = torch.cat(outputs, dim=1)

        return logits, probs, mus


# ============================================================
# 4. 原始概率型 MoE Loss
#
# L = -log sum_i p_i * N(y; mu_i, sigma^2)
# ============================================================
def moe_loss(probs, mus, y, sigma=0.5):

    # [B,1] -> [B,K]
    y_expand = y.expand_as(mus)

    # --------------------------------------------------------
    # log q_i
    #
    # q_i = N(y; mu_i, sigma^2)
    #
    # log q_i =
    # - (y-mu_i)^2 / (2 sigma^2)
    # - log(sigma)
    # - 1/2 log(2pi)
    # --------------------------------------------------------
    log_q = (
        -0.5 * ((y_expand - mus) / sigma) ** 2
        - math.log(sigma)
        - 0.5 * math.log(2 * math.pi)
    )

    # log p_i
    log_p = torch.log(probs + 1e-8)

    # --------------------------------------------------------
    # log sum_i p_i q_i
    #
    # 使用 logsumexp 保证数值稳定
    # --------------------------------------------------------
    log_mixture = torch.logsumexp(
        log_p + log_q,
        dim=1
    )

    # Negative Log Likelihood
    loss = -log_mixture.mean()

    return loss


In [2]:

# ============================================================
# 5. 构造训练数据
#
#             2x + 1, x < 0
# y =
#            -3x + 1, x >= 0
#
# 注意：
# 我们没有告诉模型：
#   x < 0 -> Expert 1
#   x >=0 -> Expert 2
#
# 这个分工必须由 MoE 自己学出来
# ============================================================

N = 1000

x = torch.linspace(-5, 5, N).reshape(-1, 1)

y = torch.where(
    x < 0,
    2 * x + 1,
    -3 * x + 1
)


# ============================================================
# 6. 创建模型
# ============================================================

model = MoE(num_experts=2)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)


# ============================================================
# 7. 训练
# ============================================================

epochs = 3000

for epoch in range(epochs):

    logits, probs, mus = model(x)

    loss = moe_loss(
        probs,
        mus,
        y,
        sigma=0.5
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if epoch % 500 == 0:
        print(
            f"epoch = {epoch:4d}, "
            f"loss = {loss.item():.6f}"
        )


epoch =    0, loss = 81.453491
epoch =  500, loss = 1.608766
epoch = 1000, loss = 0.776782
epoch = 1500, loss = 0.337591
epoch = 2000, loss = 0.250920
epoch = 2500, loss = 0.240568


In [3]:

# ============================================================
# 8. 查看两个 Expert 最终学到的函数
# ============================================================

print("\n================ Expert Parameters ================\n")

for i, expert in enumerate(model.experts):

    w = expert.fc.weight.item()
    b = expert.fc.bias.item()

    print(
        f"Expert {i}: "
        f"y = {w:.4f} * x + {b:.4f}"
    )


# ============================================================
# 9. 推理测试
#
# 这里故意选择左右两侧的点：
#
# x < 0
# x > 0
#
# 看 Router 是否选择不同 Expert
# ============================================================

test_x = torch.tensor([
    [-4.0],
    [-2.0],
    [-1.0],
    [ 1.0],
    [ 2.0],
    [ 4.0]
])

# GT
test_y = torch.where(
    test_x < 0,
    2 * test_x + 1,
    -3 * test_x + 1
)


with torch.no_grad():

    logits, probs, mus = model(test_x)

    # Router选择概率最大的Expert
    selected_expert = torch.argmax(
        probs,
        dim=1
    )

    # 根据Router选择最终预测
    final_prediction = mus.gather(
        1,
        selected_expert.unsqueeze(1)
    )



================ Expert Parameters ================

Expert 0: y = 1.9953 * x + 0.9846
Expert 1: y = -2.9945 * x + 0.9821


In [4]:

# ============================================================
# 10. 打印推理结果
# ============================================================

print("\n================ Inference ================\n")

for i in range(len(test_x)):

    print(f"x = {test_x[i].item():5.1f}")

    print(
        f"GT       = {test_y[i].item():8.3f}"
    )

    print(
        f"Expert 0 = {mus[i,0].item():8.3f}"
    )

    print(
        f"Expert 1 = {mus[i,1].item():8.3f}"
    )

    print(
        f"Router   = "
        f"[{probs[i,0].item():.4f}, "
        f"{probs[i,1].item():.4f}]"
    )

    print(
        f"Selected = Expert "
        f"{selected_expert[i].item()}"
    )

    print(
        f"Prediction = "
        f"{final_prediction[i].item():8.3f}"
    )

    print("-" * 45)


================ Inference ================

x =  -4.0
GT       =   -7.000
Expert 0 =   -6.997
Expert 1 =   12.960
Router   = [1.0000, 0.0000]
Selected = Expert 0
Prediction =   -6.997
---------------------------------------------
x =  -2.0
GT       =   -3.000
Expert 0 =   -3.006
Expert 1 =    6.971
Router   = [1.0000, 0.0000]
Selected = Expert 0
Prediction =   -3.006
---------------------------------------------
x =  -1.0
GT       =   -1.000
Expert 0 =   -1.011
Expert 1 =    3.977
Router   = [0.9989, 0.0011]
Selected = Expert 0
Prediction =   -1.011
---------------------------------------------
x =   1.0
GT       =   -2.000
Expert 0 =    2.980
Expert 1 =   -2.012
Router   = [0.0011, 0.9989]
Selected = Expert 1
Prediction =   -2.012
---------------------------------------------
x =   2.0
GT       =   -5.000
Expert 0 =    4.975
Expert 1 =   -5.007
Router   = [0.0000, 1.0000]
Selected = Expert 1
Prediction =   -5.007
---------------------------------------------
x =   4.0
GT       =  -1


## 我们期待看到什么？

首先看 Expert 参数。

理想情况下，训练结束以后可能得到：

```text
================ Expert Parameters ================

Expert 0: y =  2.0012 * x + 1.0031
Expert 1: y = -2.9987 * x + 0.9975
```

或者反过来：

```text
Expert 0: y = -3x + 1
Expert 1: y =  2x + 1
```

**Expert 编号没有任何语义，所以两种结果完全等价。**

我们没有告诉：

```text
Expert 0 学 2x+1
Expert 1 学 -3x+1
```

但它们通过竞争自己产生了：

$$
\boxed{
Expert_A\rightarrow 2x+1
}
$$

$$
\boxed{
Expert_B\rightarrow -3x+1
}
$$

这就是我们要观察的 specialization。

---

## 推理阶段更加关键

假设最终：

$$
E_0(x)\approx2x+1
$$

$$
E_1(x)\approx-3x+1
$$

那么对于：

$$
x=-2
$$

GT：

$$
y=2(-2)+1=-3
$$

可能得到：

```text
x = -2.0

GT       = -3.000

Expert 0 = -3.002
Expert 1 =  7.010

Router   = [0.9981, 0.0019]

Selected = Expert 0

Prediction = -3.002
```

Router 明显选择：

$$
Expert_0
$$

而对于：

$$
x=2
$$

GT：

$$
y=-3(2)+1=-5
$$

可能得到：

```text
x = 2.0

GT       = -5.000

Expert 0 =  5.003
Expert 1 = -4.998

Router   = [0.0023, 0.9977]

Selected = Expert 1

Prediction = -4.998
```

Router 切换到了：

$$
Expert_1
$$

因此我们最终希望验证三件事情：

$$
\boxed{
\begin{array}{c}
Expert_0\text{ 学会其中一个线性函数}\\
Expert_1\text{ 学会另一个线性函数}\\
Router\text{ 学会在 }x=0\text{ 附近切换Expert}
\end{array}
}
$$

---

## 这段代码与前面公式是一一对应的

代码中的：

```python
logits, probs = self.gate(x)
```

对应：

$$
a_i\rightarrow
p_i=Softmax(a)_i
$$

Expert：

```python
mus = torch.cat(outputs, dim=1)
```

对应：

$$
\mu_i=E_i(x)
$$

Gaussian：

```python
log_q = -0.5 * ((y_expand - mus) / sigma) ** 2 + ...
```

对应：

$$
q_i
=
\mathcal N(y;\mu_i,\sigma^2)
$$

然后：

```python
log_mixture = torch.logsumexp(
    log_p + log_q,
    dim=1
)
```

对应：

$$
\log
\sum_i
p_iq_i
$$

最终：

```python
loss = -log_mixture.mean()
```

就是：

$$
\boxed{
L
=
-\frac1B
\sum_{n=1}^{B}
\log
\left[
\sum_i
p_i(x_n)
\mathcal N
(y_n;\mu_i(x_n),\sigma^2)
\right]
}
$$

最后一句：

```python
loss.backward()
```

就会自动产生我们上一轮详细推导的：

$$
\boxed{
\frac{\partial L}{\partial a_i}
=
p_i-r_i
}
$$

以及：

$$
\boxed{
\frac{\partial L}{\partial\mu_i}
=
r_i\frac{\mu_i-y}{\sigma^2}
}
$$

所以这个小实验的重点不是拟合一个分段函数本身，而是**亲眼看到：我们没有给 Expert 分工标签，但 mixture likelihood + responsibility 机制会让 Expert 自己形成分工，同时让 Router 学会什么输入应该交给谁。**